# A2 Two-company SEC Source Probe

Formal probe for CHWY (Inventory-led) and EBAY (Marketplace / Platform).

In [1]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == 'notebooks': ROOT = ROOT.parent
pd.set_option('display.max_colwidth', 120)

In [2]:
scope = pd.read_csv(ROOT / 'data/reference/a2_probe_scope.csv')
manifest = pd.read_csv(ROOT / 'data/raw/sec/a2_probe_manifest.csv', dtype={'cik': str})
display(scope)
display(manifest[['company_id','ticker','artifact','relative_path','sha256']])

,company_id,ticker,probe_role,selection_reason,third_case_required
0,chwy,CHWY,Inventory-led E-commerce,"Inventory-owning retailer with a 52/53-week fiscal year, comparative restatements, and a distinct asset structure",0
1,ebay,EBAY,Marketplace / Platform,"Asset-light marketplace with net revenue recognition, multiple annual filing versions, and a debt-bearing balance sheet",0


,company_id,ticker,artifact,relative_path,sha256
0,chwy,CHWY,companyfacts,data/raw/sec/CIK0001766502/companyfacts.json.gz,64fb7b4c94e7fd40fe4953981110142cba7f27d841a0f0c8d4fb1a2ff813bf1d
1,chwy,CHWY,submissions,data/raw/sec/CIK0001766502/submissions.json.gz,ec18f58bea7a48bb87f5da54af7bd3af9c45f8952dfe99e4bb9f6343f55400b4
2,ebay,EBAY,companyfacts,data/raw/sec/CIK0001065088/companyfacts.json.gz,3f5ee3dcafa9ee11afc180caeaf3fbf396b9d439ece45a5970c97065fbec2d63
3,ebay,EBAY,submissions,data/raw/sec/CIK0001065088/submissions.json.gz,0b953628f45cb05478bebbd7a0bbdfa9acc9565f66b3d47209f5f08599cc453b


In [3]:
field_probe = pd.read_csv(ROOT / 'data/processed/a2_field_probe.csv')
display(field_probe[['ticker','canonical_field','normalized_source_tags','normalized_units','duration_days','multiple_filing_versions','shared_mapping_status','company_override_candidate']])

,ticker,canonical_field,normalized_source_tags,normalized_units,duration_days,multiple_filing_versions,shared_mapping_status,company_override_candidate
0,CHWY,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,USD,363.0,1,shared_direct,none_observed_in_probe
1,CHWY,net_income,NetIncomeLoss,USD,363.0,1,shared_direct,none_observed_in_probe
2,CHWY,total_assets,Assets,USD,NaN,1,shared_direct,none_observed_in_probe
3,CHWY,total_equity,StockholdersEquity,USD,NaN,1,shared_direct,none_observed_in_probe
4,CHWY,cash_and_equivalents,CashAndCashEquivalentsAtCarryingValue,USD,NaN,1,shared_direct,none_observed_in_probe
5,CHWY,inventory,InventoryNet,USD,NaN,1,conditional_not_applicable,none_observed_in_probe
6,CHWY,current_assets,AssetsCurrent,USD,NaN,1,shared_direct,none_observed_in_probe
7,CHWY,current_liabilities,LiabilitiesCurrent,USD,NaN,1,shared_direct,none_observed_in_probe
8,CHWY,total_debt,NaN,NaN,NaN,0,company_review_required,filing_verification_or_documented_aggregation
9,CHWY,operating_cash_flow,NetCashProvidedByUsedInOperatingActivities,USD,363.0,1,shared_direct,none_observed_in_probe


In [4]:
latest = pd.read_csv(ROOT / 'data/processed/a2_latest_restated_sample.csv')
conflicts = pd.read_csv(ROOT / 'data/processed/a2_concept_conflicts_sample.csv')
display(latest[['ticker','fiscal_year','canonical_field','source_tag','accession','filing_date','value_standardized']].head(20))
display(conflicts.head(20))

,ticker,fiscal_year,canonical_field,source_tag,accession,filing_date,value_standardized
0,CHWY,2021,capital_expenditure,PaymentsToAcquireProductiveAssets,0001766502-24-000014,2024-03-20,183.186
1,CHWY,2021,cash_and_equivalents,CashAndCashEquivalentsAtCarryingValue,0001766502-23-000011,2023-03-22,603.079
2,CHWY,2021,current_assets,AssetsCurrent,0001766502-23-000011,2023-03-22,1323.532
3,CHWY,2021,current_liabilities,LiabilitiesCurrent,0001766502-23-000011,2023-03-22,1644.879
4,CHWY,2021,inventory,InventoryNet,0001766502-23-000011,2023-03-22,560.430
5,CHWY,2021,net_income,NetIncomeLoss,0001766502-24-000014,2024-03-20,-75.207
6,CHWY,2021,operating_cash_flow,NetCashProvidedByUsedInOperatingActivities,0001766502-24-000014,2024-03-20,191.743
7,CHWY,2021,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,0001766502-24-000014,2024-03-20,8967.407
8,CHWY,2021,total_assets,Assets,0001766502-23-000011,2023-03-22,2086.281
9,CHWY,2021,total_equity,StockholdersEquity,0001766502-25-000014,2025-03-26,-39.620


,company_id,period_end,canonical_field,winning_tag,discarded_tag,winning_value,discarded_value,relative_difference,resolution_rule,conflict_severity,winning_accession,discarded_accession,winning_filing_date,discarded_filing_date
0,chwy,2022-01-30,net_income,NetIncomeLoss,NetIncomeLoss,-75.207,-73.817,0.018482,latest valid filing then configured source-tag priority,medium,0001766502-24-000014,0001766502-23-000011,2024-03-20,2023-03-22
1,chwy,2022-01-30,net_income,NetIncomeLoss,NetIncomeLoss,-75.207,-73.817,0.018482,latest valid filing then configured source-tag priority,medium,0001766502-24-000014,0001766502-22-000008,2024-03-20,2022-03-29
2,chwy,2022-01-30,operating_cash_flow,NetCashProvidedByUsedInOperatingActivities,NetCashProvidedByUsedInOperatingActivities,191.743,191.739,0.000021,latest valid filing then configured source-tag priority,low,0001766502-24-000014,0001766502-23-000011,2024-03-20,2023-03-22
3,chwy,2022-01-30,operating_cash_flow,NetCashProvidedByUsedInOperatingActivities,NetCashProvidedByUsedInOperatingActivities,191.743,191.739,0.000021,latest valid filing then configured source-tag priority,low,0001766502-24-000014,0001766502-22-000008,2024-03-20,2022-03-29
4,chwy,2022-01-30,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,RevenueFromContractWithCustomerExcludingAssessedTax,8967.407,8890.773,0.008546,latest valid filing then configured source-tag priority,medium,0001766502-24-000014,0001766502-23-000011,2024-03-20,2023-03-22
5,chwy,2022-01-30,revenue,RevenueFromContractWithCustomerExcludingAssessedTax,RevenueFromContractWithCustomerExcludingAssessedTax,8967.407,8890.773,0.008546,latest valid filing then configured source-tag priority,medium,0001766502-24-000014,0001766502-22-000008,2024-03-20,2022-03-29
6,chwy,2022-01-30,total_equity,StockholdersEquity,StockholdersEquity,-39.620,-39.620,0.000000,latest valid filing then configured source-tag priority,low,0001766502-25-000014,0001766502-24-000014,2025-03-26,2024-03-20
7,chwy,2022-01-30,total_equity,StockholdersEquity,StockholdersEquity,-39.620,14.736,1.371933,latest valid filing then configured source-tag priority,high,0001766502-25-000014,0001766502-23-000011,2025-03-26,2023-03-22
8,chwy,2022-01-30,total_equity,StockholdersEquity,StockholdersEquity,-39.620,14.736,1.371933,latest valid filing then configured source-tag priority,high,0001766502-25-000014,0001766502-22-000008,2025-03-26,2022-03-29
9,chwy,2023-01-29,cash_and_equivalents,CashAndCashEquivalentsAtCarryingValue,CashAndCashEquivalentsAtCarryingValue,331.641,330.441,0.003618,latest valid filing then configured source-tag priority,low,0001766502-24-000014,0001766502-23-000011,2024-03-20,2023-03-22


In [5]:
audit = json.loads((ROOT / 'data/processed/a2_source_probe_audit.json').read_text())
display(pd.Series(audit['checks'], name='passed'))
audit

probe_roles_exact                  True
single_extraction_entry_exists     True
raw_manifest_complete              True
raw_checksums_valid                True
extraction_error_log_clear         True
concept_map_executable             True
all_first_round_fields_observed    True
filing_metadata_complete           True
annual_flow_durations_valid        True
latest_winners_unique              True
conflict_log_schema_complete       True
capex_sign_rule_explicit           True
ocf_sign_rule_explicit             True
a3_scan_design_written             True
probe_report_written               True
probe_notebook_written             True
third_case_not_added               True
Name: passed, dtype: bool

{'generated_on': '2026-08-05',
 'stage': 'A2 Two-company Source Probe',
 'status': 'Done',
 'probe_companies': ['CHWY', 'EBAY'],
 'raw_artifact_count': 4,
 'annual_fact_sample_rows': 155,
 'latest_winner_rows': 60,
 'conflict_rows': 38,
 'field_probe_rows': 22,
 'canonical_source_conclusion': 'suitable_for_a3_with_validation',
 'third_distress_case': 'not_required',
 'gate1_status': 'pending_a3',
 'checks': {'probe_roles_exact': True,
  'single_extraction_entry_exists': True,
  'raw_manifest_complete': True,
  'raw_checksums_valid': True,
  'extraction_error_log_clear': True,
  'concept_map_executable': True,
  'all_first_round_fields_observed': True,
  'filing_metadata_complete': True,
  'annual_flow_durations_valid': True,
  'latest_winners_unique': True,
  'conflict_log_schema_complete': True,
  'capex_sign_rule_explicit': True,
  'ocf_sign_rule_explicit': True,
  'a3_scan_design_written': True,
  'probe_report_written': True,
  'probe_notebook_written': True,
  'third_case_not_adde

The probe keeps every filing-level version, validates expected units and flow durations before winner selection, and logs discarded values. Raw `fy` identifies the filing context for comparative facts, so the project fiscal year is mapped from the period end and issuer fiscal calendar. No third distress case is added at A2; event-quarter feasibility is measured across the full event pool in A3.